# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [29]:
#Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier,  GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier 

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
spaceship.shape

(8693, 14)

In [4]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

In [5]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [6]:
spaceship.dropna(inplace=True)

In [7]:
# Cabin is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}
spaceship["Cabin"] = spaceship["Cabin"].str[0]


In [8]:
# Drop PassengerId and Name
spaceship.drop(["PassengerId", "Name"], axis=1, inplace=True)

In [9]:
# For non-numerical columns, do dummies.
categorical_spaceship = spaceship.select_dtypes('object')
spaceship_df = pd.get_dummies(spaceship, columns = ["HomePlanet", "CryoSleep", "Cabin", "Destination",  "VIP"],  dtype=int, drop_first = True)
spaceship_df

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,1,0,0,1,0,0,0,0,0,0,0,1,0
1,24.0,109.0,9.0,25.0,549.0,44.0,True,0,0,0,0,0,0,0,1,0,0,0,1,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,1,0,0,0,0,0,0,0,0,0,0,1,1
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,1,0,0,0,0,0,0,0,0,0,0,1,0
4,16.0,303.0,70.0,151.0,565.0,2.0,True,0,0,0,0,0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,41.0,0.0,6819.0,0.0,1643.0,74.0,False,1,0,0,0,0,0,0,0,0,0,0,0,1
8689,18.0,0.0,0.0,0.0,0.0,0.0,False,0,0,1,0,0,0,0,0,1,0,1,0,0
8690,26.0,0.0,0.0,1872.0,1.0,0.0,True,0,0,0,0,0,0,0,0,1,0,0,1,0
8691,32.0,0.0,1049.0,0.0,353.0,3235.0,False,1,0,0,0,0,0,1,0,0,0,0,0,0


In [10]:
# prepare data
X = spaceship_df.drop('Transported', axis=1)
y = spaceship_df["Transported"]
X

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,1,0,0,1,0,0,0,0,0,0,0,1,0
1,24.0,109.0,9.0,25.0,549.0,44.0,0,0,0,0,0,0,0,1,0,0,0,1,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,1,0,0,0,0,0,0,0,0,0,0,1,1
3,33.0,0.0,1283.0,371.0,3329.0,193.0,1,0,0,0,0,0,0,0,0,0,0,1,0
4,16.0,303.0,70.0,151.0,565.0,2.0,0,0,0,0,0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,41.0,0.0,6819.0,0.0,1643.0,74.0,1,0,0,0,0,0,0,0,0,0,0,0,1
8689,18.0,0.0,0.0,0.0,0.0,0.0,0,0,1,0,0,0,0,0,1,0,1,0,0
8690,26.0,0.0,0.0,1872.0,1.0,0.0,0,0,0,0,0,0,0,0,1,0,0,1,0
8691,32.0,0.0,1049.0,0.0,353.0,3235.0,1,0,0,0,0,0,1,0,0,0,0,0,0


**Perform Train Test Split**

In [13]:
# Split train/test
scaler = StandardScaler() 
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=0)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [33]:
knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

print(classification_report(y_test, y_pred_knn))
print(f"KNN Accuracy is {accuracy_score(y_pred_knn, y_test): .2f}")

              precision    recall  f1-score   support

       False       0.77      0.81      0.79       661
        True       0.80      0.76      0.78       661

    accuracy                           0.78      1322
   macro avg       0.79      0.78      0.78      1322
weighted avg       0.79      0.78      0.78      1322

KNN Accuracy is  0.78


- Bagging and Pasting

Our current best model is a KNN  with R-Squared of 67.25%, lets see how ensembles works

In [34]:
bagging_knn = BaggingClassifier(
    estimator=KNeighborsClassifier(n_neighbors=10),
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,      
    n_jobs=-1,
    random_state=0
)

bagging_knn.fit(X_train, y_train)
y_pred_test_bag = bagging_knn.predict(X_test)

print(classification_report(y_test, y_pred_test_bag))
print(f"Accuracy Bagging (KNN) {accuracy_score(y_pred_test_bag, y_test): .2f}")

              precision    recall  f1-score   support

       False       0.80      0.75      0.78       661
        True       0.77      0.82      0.79       661

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.78      1322
weighted avg       0.79      0.79      0.78      1322

Accuracy Bagging (KNN)  0.79


- Random Forests

In [30]:
forest = RandomForestClassifier(random_state=42)

forest.fit(X_train, y_train)
y_pred_test_rf = forest.predict(X_test)

print(f"Accuracy Random Forest (KNN) {accuracy_score(y_pred_test_rf, y_test): .2f}")
print(classification_report(y_test, y_pred_test_rf))


Accuracy Random Forest (KNN)  0.79
              precision    recall  f1-score   support

       False       0.78      0.80      0.79       661
        True       0.80      0.78      0.79       661

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



In [36]:
forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=0
)

forest.fit(X_train, y_train)
y_pred_test_rf = forest.predict(X_test)


print(classification_report(y_test, y_pred_test_rf))
print(f"Accuracy Bagging (KNN) {accuracy_score(y_pred_test_rf, y_test): .2f}")

              precision    recall  f1-score   support

       False       0.78      0.80      0.79       661
        True       0.80      0.78      0.79       661

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322

Accuracy Bagging (KNN)  0.79


- Gradient Boosting

In [37]:
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb.fit(X_train, y_train)
y_pred_test_gb = gb.predict(X_test)


print(classification_report(y_test, y_pred_test_gb))
print(f"Accuracy Bagging (KNN) {accuracy_score(y_pred_test_gb, y_test): .2f}")

              precision    recall  f1-score   support

       False       0.82      0.74      0.78       661
        True       0.76      0.83      0.80       661

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322

Accuracy Bagging (KNN)  0.79


- Adaptive Boosting

In [38]:
ada = AdaBoostClassifier(
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

ada.fit(X_train, y_train)
y_pred_test_ada = ada.predict(X_test)

print(classification_report(y_test, y_pred_test_ada))
print(f"Accuracy Bagging (KNN) {accuracy_score(y_pred_test_ada, y_test): .2f}")

              precision    recall  f1-score   support

       False       0.77      0.79      0.78       661
        True       0.79      0.76      0.77       661

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322

Accuracy Bagging (KNN)  0.78


Which model is the best and why?

## Interpretation :

The best model based on the results is the Random Forest Classifier.
Although several models achieved similar accuracy (~0.79), Random Forest provides the most balanced performance across both classes, 
is robust to noise and scaling, and offers stable predictions without requiring heavy tuning. It also provides interpretable feature importance, 
making it the most reliable and practical choice for this dataset.
